# Проект: ✎ название

> **Как работать с шаблоном** (эту врезку удалите перед сдачей)
>
> 1. Скопируйте файл и переименуйте по проекту, например `ch05-project.ipynb`. Положите его в `site/notebooks` — тогда путь к данным `../data` в первой ячейке подойдёт без правок. Пути к файлам — только относительные: блокнот должен запускаться на любом компьютере.
> 2. Заполните разделы по порядку: цель → данные → предобработка → анализ и модели → выводы. Всё, что помечено ✎, замените своим текстом. Код-пример посчитан на `daily_sales.csv` — замените его кодом своего проекта.
> 3. После каждого раздела — markdown-ячейка **«Вывод»**: что увидели (с числами из вывода кода) и что это значит для «Зерна». Код без вывода — это черновик, а не отчёт.
> 4. Переменные с результатами называйте так, как просят задачи проекта в браузере (`monthly`, `late_pct`, `best_t` …): по этим именам скрипт самопроверки сверит ваши числа с эталонами.
> 5. Перед сдачей: **Kernel → Restart Kernel and Run All Cells**, затем в терминале, в папке блокнота: `python check_notebook.py my.ipynb --project chNN/<урок>` — например, `--project ch04/09-project`. Скрипт выполнит блокнот с чистого ядра, перечислит, что поправить в оформлении, и проверит переменные блокнота проверками задач урока: ✓ или ✗ с подсказкой по каждой. Если блокнот лежит вне папки курса, добавьте `--course ПАПКА_КУРСА`.
>
> **Чек-лист самопроверки**
>
> - [ ] блокнот выполняется сверху вниз с чистого ядра без ошибок;
> - [ ] `check_notebook.py … --project` — задачи урока сошлись с эталонами (✓);
> - [ ] первая ячейка — импорты, версии библиотек и `RANDOM_STATE`; случайность везде через `random_state=RANDOM_STATE`;
> - [ ] пути к данным относительные, нет `C:\…` и `/Users/…`;
> - [ ] нет закомментированного кода и отладочных ячеек;
> - [ ] после каждого раздела есть вывод с числами, а в конце — ответ на вопрос из «Цели»;
> - [ ] у графиков есть заголовок и подписи осей;
> - [ ] модель сравнивается с простым ориентиром (среднее, самый частый класс, «как вчера»).

In [ ]:
# Импорты, версии и общие настройки — всё в одном месте, в начале блокнота
import sys
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:                       # sklearn нужен только проектам с моделями (с главы 6)
    import sklearn
    from sklearn.dummy import DummyRegressor
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error
    from sklearn.model_selection import train_test_split
except ImportError:
    sklearn = None

RANDOM_STATE = 42          # одно число для всей случайности: разбиения, модели, выборки
# папка с данными: data/ рядом с блокнотом (свой проект) или ../data (блокнот в папке курса)
DATA = next((p for p in (Path('data'), Path('../data')) if p.exists()), Path('data'))

pd.set_option('display.max_columns', 30)
print('Python', sys.version.split()[0], '| данные:', DATA.resolve())
for lib in (pd, np, matplotlib, sklearn):
    if lib is None:
        continue
    print(f'{lib.__name__:<11}{lib.__version__}')

## 1. Цель

✎ **Бизнес-вопрос.** Какой вопрос «Зерна» решает проект и кто примет решение по результату.

✎ **Что будет результатом.** Таблица, график, модель, рекомендация — и в каком виде её получит заказчик.

✎ **Как поймём, что получилось.** Метрика и порог: например, «ошибка прогноза меньше, чем у прогноза „как в среднем“».

✎ **Главное (допишите в конце работы).** 2–3 фразы для руководителя: ответ на вопрос с цифрой масштаба и что предлагаем. Подробности и журнал очистки — ниже, в своих разделах.

## 2. Данные

✎ Откуда данные, за какой период, что описывает одна строка, какие столбцы важны для вопроса.

In [ ]:
df = pd.read_csv(DATA / 'daily_sales.csv', parse_dates=['date'])   # ✎ замените на файл своего проекта
print(df.shape)
df.info()

In [ ]:
df.head()

**Вывод по данным.** ✎ Сколько строк и столбцов, за какой период, какие типы надо поправить, что вызывает вопросы.

## 3. Предобработка

Сначала смотрим на проблемы, потом исправляем — и каждое исправление объясняем.

In [ ]:
print('Пропуски по столбцам:')
print(df.isna().sum()[lambda s: s > 0])
print('Полных дублей строк:', df.duplicated().sum())
df.describe()

In [ ]:
clean = df.drop_duplicates().copy()
print(f'Было строк: {len(df)}, стало: {len(clean)}')

**Вывод по предобработке.** ✎ Что нашли (пропуски, дубли, выбросы, типы), что сделали с каждой проблемой и почему, сколько строк осталось.

## 4. Анализ и модели

✎ Коротко: какие гипотезы проверяем и какие модели сравниваем.

In [ ]:
by_format = clean.groupby('format')['revenue_k'].mean().sort_values()
ax = by_format.plot(kind='barh', figsize=(7, 3))
ax.set_title('Средняя дневная выручка по формату точки')
ax.set_xlabel('тыс. руб. в день')
ax.set_ylabel('')
plt.show()
by_format.round(1)

In [ ]:
# Модели — в проектах с главы 6; в проектах глав 2–4 эту ячейку удалите.
features = ['temp', 'rain', 'holiday', 'promo']
X_train, X_test, y_train, y_test = train_test_split(
    clean[features], clean['revenue_k'], test_size=0.25, random_state=RANDOM_STATE)

results = []
for name, model in [('среднее (ориентир)', DummyRegressor()), ('линейная регрессия', LinearRegression())]:
    model.fit(X_train, y_train)
    results.append({'модель': name, 'MAE на тесте': mean_absolute_error(y_test, model.predict(X_test))})
pd.DataFrame(results).round(2)

**Вывод по анализу.** ✎ Что показали графики и таблицы, какая модель лучше ориентира и насколько — в единицах бизнеса.

## 5. Выводы и рекомендации

✎ 3–5 пунктов:

1. Ответ на вопрос из раздела «Цель» — с числами.
2. Что рекомендуем сделать «Зерну» и какой эффект ожидаем.
3. Ограничения: чего данные не показывают, где выводы могут не сработать.
4. Что проверить или сделать дальше.